In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "volter2019chimpanzees")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Voelter_2019_exp_1_retest_error_data_GLMM S01_ProcB_CHSK.csv")
complete_path_2 = os.path.join(original_data_pathway, "Voelter_2019_exp_1_trial data_GLMM02_ProcB_CHSK.csv")
complete_path_3 = os.path.join(original_data_pathway, "Voelter_2019_exp_1error data GLMM01_ProcB_CHSK.csv")
complete_path_4 = os.path.join(original_data_pathway, "Voelter_2019_exp_2_trial data_GLMM03 and S02_ProcB_CHSK.csv")


out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)
df3 = pd.read_csv(complete_path_3)
df4 = pd.read_csv(complete_path_4)
experiment_import = [[df1, 'retest_error_data_GLMM_S01', '1'],
                    [df2, 'trial_data_GLMM02', '1'],
                    [df3, 'error_data_GLMM01', '1'],
                    [df4, 'trial_data_GLMM03_and_S02', '2']]
for x,y,k in experiment_import:
    x['model_name'] = y
    x['experiment'] = k

In [3]:
data_frames=[df1, df2, df3, df4]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"subject": "ape",
        "sex":"sex_original",
        "species":"species_original",
        "trial.id":"trial_id",
        "choice.nr.within.trial":"choice_nr_within_trial",
        "choice.where":"choice_where",
        "choice.correct":"choice_correct",
        "last.choiceany.error.earlier":"last_choice_any_error_earlier",
        "any.error.earlier":"any_error_earlier",
        "cup.id":"cup_id",
        "choice.id":"choice_id",
        "error.ny":"error_ny",
        "number.cups":"number_cups",
        "cup.is.edge":"cup_is_edge",
        "cup.is.edge.code":"cup_is_edge_code",
        "z.lag.from.last.choice":"z_lag_from_last_choice",
        "any.code":"any_code",
        "z.number_boxes":"z_number_boxes",
        "z.trial_number":"z_trial_number",
        "z.age":"z_age",
        "lag.from.last.choice":"lag_from_last_choice",
        "choice.latency":"choice_latency",
        "absence.latency":"absence_latency",
        "feature-space strategy":"feature_space_strategy",
        "same boxes strategy":"same_boxes_strategy",
        "last.choice":"last_choice"}, inplace=True)
    x['study_id']="volter2019chimpanzees"
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)


In [4]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase_chimpanzee_groups.csv")
apedf = pd.read_csv(comp_path_ape_info)   
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')
# fulldf.columns
fulldf.rename(columns={"ape": "participant", "age":"age_in_years"}, inplace=True)

fulldf['phase'].replace("feature+space", 'feature_space', inplace=True, regex=False)
fulldf['phase'].replace("fs", 'feature_space', inplace=True, regex=False)

In [5]:
fulldf.rename(columns={"phase": "phase_temp", "subgroup":"species_subgroup"}, inplace=True)
fulldf.loc[fulldf.phase_temp == 'f', ['phase']] = 'feature'
fulldf.loc[fulldf.phase_temp == 's', ['phase']] = 'space'
fulldf.loc[fulldf.phase_temp == 'feature_space', ['phase']] = 'feature_space'

In [6]:
fulldf=fulldf[[ 'study_id', 'experiment', 'model_name',
        'participant','age_in_years', 'z_age','sex','species', 'species_subgroup', 'phase','phase_temp',
        'session_within_number_of_boxes', 'session_within_phase', 'session_overall',
       'trial_number', 'z_trial_number','trial_id', 'condition', 'session_within_condition', 
       'choice_nr_within_trial', 'order','order2',  'number_boxes','choice_where',
       'choice_correct', 'last_choice', 'any_error_earlier', 'cup_id',
       'choice_id', 'error_ny', 'number_cups', 'cup_is_edge',
       'cup_is_edge_code', 'z_lag_from_last_choice', 'any_code',
       'z_number_boxes',   'lag_from_last_choice',
       'trial_correct', 'number_redundant_searched', 'first_choice_feature',
       'first_choice', 'second_choice_feature', 'second_choice',
       'second_choice_correct', 'third_choice_feature', 'third_choice',
       'third_choice_correct', 'fourth_choice_feature', 'fourth_choice',
       'fourth_choice_correct', 'fifth_choice_feature', 'fifth_choice',
       'fifth_choice_correct', 'sixth_choice_feature', 'sixth_choice',
       'sixth_choice_correct', 'choice_latency',
         'absence_latency',
       'p1p2_trial_correct',
       'redundant_searches', 'p1_trial_correct', 'p1_redundant_searches',
       'p1_1st', 'p1_2nd', 'p1_2nd_correct', 'p1_3rd', 'p1_3rd_correct',
       'p1_4th', 'p1_4th_correct', 'p2_trial_correct', 'p2_redundant_searches',
       'p2_1st', 'p2_2nd', 'p2_2nd_correct', 'p2_3rd', 'p2_3rd_correct',
       'p2_4th', 'p2_4th_correct', 'feature_space_strategy',
       'same_boxes_strategy']]


In [7]:
for index in range(1,3):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'volter2019chimpanzees_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'volter2019chimpanzees_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)